In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


# Load Dataset
df = pd.read_csv("Mall_Customers.csv")

print("First Five Records\n")
print(df.head())

print("\nDataset Information\n")
print(df.info())

print("\nSummary Statistics\n")
print(df.describe())

# Numerical Features
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Categorical Features
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)


print("\nMissing Values")
print(df.isnull().sum())

# Remove CustomerID
df = df.drop("CustomerID", axis=1)

# Encode Gender
if "Gender" in df.columns:
    encoder = LabelEncoder()
    df["Gender"] = encoder.fit_transform(df["Gender"])

# Standardize Data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)


# Elbow Method
wcss = []

for i in range(1, 11):
    model = KMeans(
        n_clusters=i,
        random_state=42,
        n_init=10
    )
    model.fit(scaled_data)
    wcss.append(model.inertia_)

# Choose Optimal K
k = 5

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(scaled_data)

df["Cluster"] = clusters

# PCA
pca = PCA(n_components=2)

pca_data = pca.fit_transform(scaled_data)

pca_df = pd.DataFrame(
    pca_data,
    columns=["PC1", "PC2"]
)

pca_df["Cluster"] = clusters


# Elbow Curve
plt.figure(figsize=(7,5))
plt.plot(range(1,11), wcss, marker='o')
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.grid(True)
plt.show()

# Customer Clusters
plt.figure(figsize=(7,5))

plt.scatter(
    df["Annual Income (k$)"],
    df["Spending Score (1-100)"],
    c=clusters,
    cmap="viridis"
)

plt.title("Customer Segments")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score")
plt.show()

# PCA Visualization
plt.figure(figsize=(7,5))

plt.scatter(
    pca_df["PC1"],
    pca_df["PC2"],
    c=pca_df["Cluster"],
    cmap="viridis"
)

plt.title("PCA Visualization of Customer Clusters")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

print("\nObservations\n")

print("1. The Elbow Method indicates that K = 5 is the optimal number of clusters.")

print("2. PCA reduces the dataset into two principal components while preserving most of the important information, making visualization easier.")

print("3. The clusters represent customers with different income and spending habits, such as high-income/high-spending and low-income/low-spending groups.")

print("4. These customer groups can help businesses design personalized marketing campaigns.")



print("\nCluster-wise Mean Values\n")

print(df.groupby("Cluster").mean())


print("\nConclusion\n")

print("""
This project used the K-Means clustering algorithm to segment
mall customers based on demographic and spending behavior.
After preprocessing and feature scaling, the Elbow Method
identified five optimal customer groups. PCA reduced the
high-dimensional dataset into two principal components,
allowing clear visualization of the customer segments. These
clusters help businesses understand purchasing behavior and
create targeted marketing campaigns, loyalty programs, and
personalized promotions. One limitation of K-Means is that the
number of clusters must be selected in advance and the
algorithm is sensitive to outliers. An important advantage of
PCA is that it reduces dimensionality while retaining most of
the important information, making visualization and analysis
much easier.
""")